# Executive Inbox Submission Runs

This notebook is the judge-facing Jupyter version of the Executive Inbox training workflow.

- It uses the local notebook environment for the benchmark runs.
- It can also show the deployed OpenEnv HF Space verification.
- The full minimal training script is shown in-cell so judges can inspect the code.


In [14]:
# No W&B setup is required for the submission notebook rerun.
# This cell simply sets the project root and keeps the notebook self-contained.
import os

os.environ.setdefault("WANDB_PROJECT", "openenv-executive-inbox")
print("W&B logging is disabled in this notebook rerun.")


W&B logging is disabled in this notebook rerun.


In [15]:
import os
from pathlib import Path

%cd /home/jovyan/OpenEnv

os.environ.setdefault("WANDB_PROJECT", "openenv-executive-inbox")

print("Repo:", Path.cwd())
print("W&B logging disabled for notebook rerun.")


/home/jovyan/OpenEnv
Repo: /home/jovyan/OpenEnv
W&B logging disabled for notebook rerun.


In [16]:
# Optional: show the exact minimal training script used for the benchmark.
from pathlib import Path

script_path = Path("scripts/executive_inbox_unsloth_reinforce.py")
print(script_path.read_text())


#!/usr/bin/env python3
"""
Standalone REINFORCE training script for OpenEnv Executive Inbox with Unsloth.

Designed to mirror the notebook flow but run unattended overnight.
"""

from __future__ import annotations

import argparse
import contextlib
import datetime as dt
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Optional

# Must be set before importing unsloth.
os.environ.setdefault("UNSLOTH_ENABLE_FLEX_ATTENTION", "0")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(line_buffering=True)

import torch
from unsloth import FastLanguageModel

AVAILABLE_TIMES = ["9:00 AM", "10:00 AM", "1:00 PM", "2:00 PM", "3:00 PM", "4:00 PM"]


def setup_openenv_paths() -> Path:
    """Resolve repo root and add OpenEnv paths to sys.path."""
    script_dir = Path(__file__).resolve().parent
    cwd = Path.cwd().resolve()
    candidates = [
      

In [17]:
# Optional: verify the deployed OpenEnv HF Space before submission.
!python scripts/verify_executive_inbox_space.py


Connecting to https://hoony-executive-inbox.hf.space
reset: Environment initialized.
read_inbox: [{"id": "e_xj2m", "sender": "jessica.pm@company.com", "subject": "Re: Project Alpha timeline"}, {"id": "e_8euo", "sender": "head.legal@company.com", "subject": "Quick legal wording pass"}, {"id": "e_q3ud", "sender": "scheduling@hvac-pros.com", "subject": "Reminder: Appointment at 9:00 AM"}, {"id"...
read_inbox(email): {"id": "e_8euo", "sender": "head.legal@company.com", "subject": "Quick legal wording pass", "date": "Today 09:00 AM", "body": "Can you glance at the wording later this afternoon? No immediate action needed."}
get_calendar: [{"id": "c_xzd6", "time": "1:00 PM", "title": "Flight Options (Do not schedule)", "participants": ["nanny@care.com"]}, {"id": "c_rpgj", "time": "9:00 AM", "title": "Anniversary Dinner", "participants": ["alerts@delta.com"]}, {"id": "c_0wf7", "time": "2:00 PM", "title": "All-Hands Rehearsal", "par...
state: {"step_count": 0, "conflict_resolved": false, "parti

In [18]:
# Run 1: curriculum training run.
!python scripts/executive_inbox_unsloth_reinforce.py \
  --episodes 20 \
  --max-steps 12 \
  --num-conflicts 1 \
  --learning-rate 2e-4 \
  --policy-mode candidate \
  --run-name curriculum-train-1conf-e20 \
  --save-dir /home/jovyan/OpenEnv/artifacts/smollm_curriculum_1conf_e20


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Repo root: /home/jovyan/OpenEnv
Local logs: /home/jovyan/OpenEnv/logs/executive_inbox_rl/20260308-195436-curriculum-train-1conf-e20
Loading model: unsloth/smollm2-1.7b-instruct
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth 2026.3.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.
Model loaded on cuda
CUDA memory after load: 0.98 GiB allocated / 1.00 GiB reserved
Episode 1/20 | Steps: 7 | Reward: 1.38 | Loss: -0.5015

In [19]:
# Run 2: held-out base evaluation.
!python scripts/executive_inbox_unsloth_reinforce.py \
  --eval-only \
  --episodes 20 \
  --max-steps 12 \
  --num-conflicts 1 \
  --seed-offset 1000 \
  --no-expert-fallback \
  --policy-mode candidate \
  --run-name curriculum-eval-base-1conf


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Repo root: /home/jovyan/OpenEnv
Local logs: /home/jovyan/OpenEnv/logs/executive_inbox_rl/20260308-195710-curriculum-eval-base-1conf
Loading model: unsloth/smollm2-1.7b-instruct
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth 2026.3.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.
Model loaded on cuda
CUDA memory after load: 0.98 GiB allocated / 1.00 GiB reserved
Episode 1/20 | Steps: 9 | Reward: 1.26 | Loss: -0.9046

In [20]:
# Run 3: held-out trained evaluation.
!python scripts/executive_inbox_unsloth_reinforce.py \
  --model-name /home/jovyan/OpenEnv/artifacts/smollm_curriculum_1conf_e20 \
  --eval-only \
  --episodes 20 \
  --max-steps 12 \
  --num-conflicts 1 \
  --seed-offset 1000 \
  --no-expert-fallback \
  --policy-mode candidate \
  --run-name curriculum-eval-trained-1conf


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Repo root: /home/jovyan/OpenEnv
Local logs: /home/jovyan/OpenEnv/logs/executive_inbox_rl/20260308-195822-curriculum-eval-trained-1conf
Loading model: /home/jovyan/OpenEnv/artifacts/smollm_curriculum_1conf_e20
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth 2026.3.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.
Model loaded on cuda
CUDA memory after load: